# Company_X Micro-Fulfillment Hub Demand Forecasting
## Dual LightGBM + Bias Calibration | Final Submission

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import os, time, warnings
warnings.filterwarnings('ignore')
T0 = time.time()

INPUT_DIR = "datasets"
for root, dirs, files in os.walk("/kaggle/input"):
    if "orders_train.csv" in files:
        INPUT_DIR = root; break

train = pd.read_csv(f"{INPUT_DIR}/orders_train.csv", parse_dates=["Date"])
test  = pd.read_csv(f"{INPUT_DIR}/orders_test.csv",  parse_dates=["Date"])
hub   = pd.read_csv(f"{INPUT_DIR}/hub_metadata.csv")
print(f"Train {train.shape}, Test {test.shape}")

In [ ]:
def build_features(train_df, test_df, hub_df, cutoff):
    h = hub_df.copy()
    h["CompetitorDistance"] = h["CompetitorDistance"].fillna(h["CompetitorDistance"].median())
    h["LogCompetitorDistance"] = np.log1p(h["CompetitorDistance"]).astype("float32")
    h["HasCompetitor"] = hub_df["CompetitorDistance"].notna().astype("int8")
    h["CompetitorOpenSinceMonth"] = h["CompetitorOpenSinceMonth"].fillna(1).astype("int8")
    h["CompetitorOpenSinceYear"]  = h["CompetitorOpenSinceYear"].fillna(1900).astype("int16")
    h["LoyaltyProgramSinceWeek"]  = h["LoyaltyProgramSinceWeek"].fillna(0).astype("int8")
    h["LoyaltyProgramSinceYear"]  = h["LoyaltyProgramSinceYear"].fillna(1900).astype("int16")

    full = pd.concat([train_df, test_df], ignore_index=True)
    full = full.merge(h, on="HubID", how="left").sort_values(["HubID","Date"]).reset_index(drop=True)

    full["Year"]       = full["Date"].dt.year.astype("int16")
    full["Month"]      = full["Date"].dt.month.astype("int8")
    full["Day"]        = full["Date"].dt.day.astype("int8")
    full["WeekOfYear"] = full["Date"].dt.isocalendar().week.astype("int8")
    full["DayOfYear"]  = full["Date"].dt.dayofyear.astype("int16")
    full["IsWeekend"]  = full["Weekday"].isin([6,7]).astype("int8")
    full["IsMonthStart"] = (full["Day"] <= 3).astype("int8")
    full["IsMonthEnd"]   = (full["Date"].dt.is_month_end | (full["Day"] >= 28)).astype("int8")
    full["Quarter"]    = full["Date"].dt.quarter.astype("int8")
    full["SinYear"]    = np.sin(2*np.pi*full["DayOfYear"]/365.25).astype("float32")
    full["CosYear"]    = np.cos(2*np.pi*full["DayOfYear"]/365.25).astype("float32")
    full["SinMonth"]   = np.sin(2*np.pi*full["Day"]/30.5).astype("float32")
    full["CosMonth"]   = np.cos(2*np.pi*full["Day"]/30.5).astype("float32")
    full["SinWeekday"] = np.sin(2*np.pi*full["Weekday"]/7.0).astype("float32")
    full["CosWeekday"] = np.cos(2*np.pi*full["Weekday"]/7.0).astype("float32")

    pb = (full["PromoActive"] != full.groupby("HubID")["PromoActive"].shift(1)).cumsum()
    full["PromoRunDay"]     = (full.groupby(["HubID", pb]).cumcount() + 1).astype("int8")
    full.loc[full["PromoActive"]==0, "PromoRunDay"] = 0
    full["IsFirstPromoDay"] = ((full["PromoActive"]==1)&(full["PromoRunDay"]==1)).astype("int8")
    full["IsLastPromoDay"]  = ((full["PromoActive"]==1)&(full.groupby("HubID")["PromoActive"].shift(-1).fillna(0)==0)).astype("int8")
    full["PromoNextDay"]    = full.groupby("HubID")["PromoActive"].shift(-1).fillna(0).astype("int8")
    full["PromoPrevDay"]    = full.groupby("HubID")["PromoActive"].shift(1).fillna(0).astype("int8")
    full["PromoNextWeek"]   = full.groupby("HubID")["PromoActive"].shift(-7).fillna(0).astype("int8")
    full["PromoPrevWeek"]   = full.groupby("HubID")["PromoActive"].shift(7).fillna(0).astype("int8")

    hol = (full["RegionalHoliday"] != "0").astype(int)
    full["DayBeforeHoliday"] = hol.groupby(full["HubID"]).shift(-1).fillna(0).astype("int8")
    full["DayAfterHoliday"]  = hol.groupby(full["HubID"]).shift(1).fillna(0).astype("int8")

    cy, cm = cutoff.year, cutoff.month
    full["CompetitorIsOpen"] = (
        (full["CompetitorOpenSinceYear"] < cy) |
        ((full["CompetitorOpenSinceYear"]==cy)&(full["CompetitorOpenSinceMonth"]<=cm))
    ).astype("int8")
    full["CompetitorMonthsOpen"] = np.clip(
        (cy - full["CompetitorOpenSinceYear"])*12 + (cm - full["CompetitorOpenSinceMonth"]), 0, 120
    ).astype("int16")
    full["LoyaltyActiveThisMonth"] = (
        (full["LoyaltyProgramSinceYear"] < cy) |
        ((full["LoyaltyProgramSinceYear"]==cy)&(full["LoyaltyProgramSinceWeek"]<=cutoff.isocalendar()[1]))
    ).astype("int8")
    loy_start = pd.to_datetime({"year":full["LoyaltyProgramSinceYear"],"month":1,"day":1})
    full["LoyaltyWeeksOpen"] = np.clip(((cutoff - loy_start).dt.days // 7).fillna(0), 0, 500).astype("int16")

    # Open streak
    full["OpenStreak"] = (full.groupby("HubID")["IsOpen"]
        .apply(lambda s: s.groupby((s!=s.shift()).cumsum()).cumcount()+1)
        .reset_index(level=0, drop=True)).astype("int16")

    # 364-day lag
    hist = full[(full["Date"] <= cutoff) & (full["IsOpen"]==1)][["HubID","Date","OrderVolume","AppSessions","SpendPerSession"]].copy()
    lag  = hist.copy()
    lag["Date"] = lag["Date"] + pd.Timedelta(days=364)
    lag = lag.rename(columns={"OrderVolume":"lag364_vol","AppSessions":"lag364_ses","SpendPerSession":"lag364_sps"})
    full = full.merge(lag[["HubID","Date","lag364_vol","lag364_ses","lag364_sps"]], on=["HubID","Date"], how="left")

    # YoY growth per hub
    old = hist[hist["Date"]<=cutoff-pd.Timedelta(days=364)].groupby("HubID")["OrderVolume"].mean()
    new = hist[hist["Date"]> cutoff-pd.Timedelta(days=182)].groupby("HubID")["OrderVolume"].mean()
    growth = (new/(old+1)).fillna(1.033).rename("h_yoy_growth")
    full = full.merge(growth.reset_index(), on="HubID", how="left")
    full["h_yoy_growth"] = full["h_yoy_growth"].fillna(1.033)

    full["has_lag364"]         = full["lag364_vol"].notna().astype("int8")
    full["lag364_vol_imputed"] = full["lag364_vol"].fillna(0).astype("float32")
    full["lag364_vol_growth"]  = (full["lag364_vol"] * full["h_yoy_growth"]).fillna(0).astype("float32")
    full["lag364_ses_imputed"] = full["lag364_ses"].fillna(0).astype("float32")
    full["lag364_sps_imputed"] = full["lag364_sps"].fillna(0).astype("float32")

    # Hub-level Bayesian stats
    for col, nm in [("OrderVolume","vol"),("AppSessions","ses"),("SpendPerSession","sps")]:
        agg = hist.groupby("HubID")[col].agg(mean="mean",median="median").reset_index()
        agg.columns = ["HubID", f"h_{nm}_mean", f"h_{nm}_med"]
        full = full.merge(agg, on="HubID", how="left")
        full[f"h_{nm}_mean"] = full[f"h_{nm}_mean"].fillna(0).astype("float32")
        full[f"h_{nm}_med"]  = full[f"h_{nm}_med"].fillna(0).astype("float32")

        # Hub x Weekday stats
        if "Weekday" in hist.columns:
            hw_agg = hist.merge(full[["HubID","Date","Weekday"]].drop_duplicates(), on=["HubID","Date"], how="left")
            hw_agg = hw_agg.groupby(["HubID","Weekday"])[col].mean().reset_index().rename(columns={col:f"hw_{nm}_mean"})
            full = full.merge(hw_agg, on=["HubID","Weekday"], how="left")
            full[f"hw_{nm}_mean"] = full[f"hw_{nm}_mean"].fillna(full[f"h_{nm}_mean"]).astype("float32")
        else:
            full[f"hw_{nm}_mean"] = full[f"h_{nm}_mean"]

    # Recency stats
    for days, sfx in [(90,"rec90"), (30,"rec30")]:
        rec = hist[hist["Date"]>cutoff-pd.Timedelta(days=days)].groupby("HubID")["OrderVolume"].mean().rename(f"h_{sfx}_vol")
        full = full.merge(rec.reset_index(), on="HubID", how="left")
        full[f"h_{sfx}_vol"] = full[f"h_{sfx}_vol"].fillna(full["h_vol_mean"]).astype("float32")

    full["h_growth_vol"]      = (full["h_rec30_vol"] / (full["h_rec90_vol"]+1)).astype("float32")
    full["hub_momentum_30_90"]= (full["h_rec30_vol"] - full["h_rec90_vol"]).astype("float32")
    full["hw_vol_std"]        = full.get("hw_vol_std", full["h_vol_med"]).fillna(0).astype("float32")
    full["hwp_vol_mean"]      = full.get("hwp_vol_mean", full["hw_vol_mean"]).fillna(0).astype("float32")
    full["hwp_vol_med"]       = full.get("hwp_vol_med",  full["h_vol_med"]).fillna(0).astype("float32")
    full["hwp_vol_std"]       = full.get("hwp_vol_std",  0.0)
    full["hw_rec90_vol"]      = full.get("hw_rec90_vol", full["h_rec90_vol"]).fillna(0).astype("float32")
    full["hwp_rec90_vol"]     = full.get("hwp_rec90_vol",full["h_rec90_vol"]).fillna(0).astype("float32")
    full["h_promo_uplift"]    = 1.0

    full["hw_ses_mean"]   = full.get("hw_ses_mean",  full["h_ses_mean"]).fillna(0).astype("float32")
    full["hwp_ses_mean"]  = full.get("hwp_ses_mean", full["h_ses_mean"]).fillna(0).astype("float32")
    full["hwp_ses_med"]   = full.get("hwp_ses_med",  full["h_ses_med"]).fillna(0).astype("float32")
    full["hwp_rec90_ses"] = full.get("hwp_rec90_ses",full["h_ses_mean"]).fillna(0).astype("float32")
    rec90_ses = hist[hist["Date"]>cutoff-pd.Timedelta(days=90)].groupby("HubID")["AppSessions"].mean().rename("h_rec90_ses")
    full = full.merge(rec90_ses.reset_index(), on="HubID", how="left")
    full["h_rec90_ses"] = full["h_rec90_ses"].fillna(0).astype("float32")

    full["hw_sps_mean"]  = full.get("hw_sps_mean",  full["h_sps_mean"]).fillna(0).astype("float32")
    full["hwp_sps_mean"] = full.get("hwp_sps_mean", full["h_sps_mean"]).fillna(0).astype("float32")
    full["hwp_sps_med"]  = full.get("hwp_sps_med",  full["h_sps_med"]).fillna(0).astype("float32")
    rec90_sps = hist[hist["Date"]>cutoff-pd.Timedelta(days=90)].groupby("HubID")["SpendPerSession"].mean().rename("h_rec90_sps")
    full = full.merge(rec90_sps.reset_index(), on="HubID", how="left")
    full["h_rec90_sps"] = full["h_rec90_sps"].fillna(0).astype("float32")

    tr_out = full[full["Date"] <= cutoff].copy()
    te_out = full[full["Date"]  > cutoff].copy()
    return tr_out, te_out

print("Feature engineering function defined.")

In [ ]:
cutoff = train["Date"].max()
print("Building features...")
tr_fe, te_fe = build_features(train, test, hub, cutoff)

tr_train = tr_fe[tr_fe["IsOpen"]==1].reset_index(drop=True)
te_val   = te_fe.reset_index(drop=True)
te_open  = (te_val["IsOpen"]==1)

dmax    = tr_train["Date"].max()
weights = (0.5**((dmax - tr_train["Date"]).dt.days / 365.0)).values.astype("float32")

FEATURES = [
    "HubID","HubFormat","AssortmentTier","RegionalHoliday","SchoolClosureFlag",
    "Weekday","PromoActive","LoyaltyProgram","Year","Month","Day",
    "WeekOfYear","DayOfYear","IsWeekend","IsMonthStart","IsMonthEnd","Quarter",
    "SinYear","CosYear","SinMonth","CosMonth","SinWeekday","CosWeekday",
    "PromoRunDay","IsFirstPromoDay","IsLastPromoDay","PromoNextDay","PromoPrevDay",
    "PromoNextWeek","PromoPrevWeek","DayBeforeHoliday","DayAfterHoliday",
    "LogCompetitorDistance","HasCompetitor","CompetitorMonthsOpen","CompetitorIsOpen",
    "LoyaltyActiveThisMonth","LoyaltyWeeksOpen","OpenStreak",
    "has_lag364","lag364_vol_imputed","lag364_vol_growth","lag364_ses_imputed","lag364_sps_imputed",
    "h_vol_mean","h_vol_med","h_vol_std","hw_vol_mean","hwp_vol_mean","hwp_vol_med","hwp_vol_std",
    "h_promo_uplift","h_rec90_vol","h_rec30_vol","hw_rec90_vol","hwp_rec90_vol",
    "h_growth_vol","hub_momentum_30_90",
    "h_ses_mean","hw_ses_mean","hwp_ses_mean","hwp_ses_med","h_rec90_ses","hwp_rec90_ses",
    "h_sps_mean","hw_sps_mean","hwp_sps_mean","hwp_sps_med","h_rec90_sps","h_yoy_growth"
]
FEATURES = [f for f in FEATURES if f in tr_train.columns]
CAT_COLS = [c for c in ["HubID","HubFormat","AssortmentTier","RegionalHoliday","SchoolClosureFlag",
                         "Weekday","PromoActive","LoyaltyProgram"] if c in FEATURES]
for c in CAT_COLS:
    tr_train[c] = tr_train[c].astype("category")
    te_val[c]   = te_val[c].astype("category")

print(f"Ready: {len(FEATURES)} features, {len(tr_train)} training rows, {te_open.sum()} test open rows")

In [ ]:
params = {
    "objective":"regression","metric":"rmse","learning_rate":0.06,
    "num_leaves":95,"min_data_in_leaf":30,"feature_fraction":0.8,
    "bagging_fraction":0.85,"bagging_freq":1,"verbose":-1,"seed":42,"n_jobs":-1
}

# Model 1: Direct LightGBM on log1p(OrderVolume)
print("Training Model 1 (Direct)...")
m1 = lgb.train(params,
    lgb.Dataset(tr_train[FEATURES], np.log1p(tr_train["OrderVolume"]), weight=weights, categorical_feature=CAT_COLS),
    num_boost_round=600)
p1 = m1.predict(te_val.loc[te_open, FEATURES])
print("Model 1 done.")

# Model 2: Two-Stage Residual (AppSessions -> Residual)
print("Training Model 2 Stage 1 (AppSessions)...")
m_ses = lgb.train(params,
    lgb.Dataset(tr_train[FEATURES], np.log1p(tr_train["AppSessions"]), weight=weights, categorical_feature=CAT_COLS),
    num_boost_round=450)
p_ses = m_ses.predict(te_val.loc[te_open, FEATURES])

print("Training Model 2 Stage 2 (Residual)...")
y_res = np.log1p(tr_train["OrderVolume"]) - np.log1p(tr_train["AppSessions"])
m_res = lgb.train(dict(params, num_leaves=63, learning_rate=0.05),
    lgb.Dataset(tr_train[FEATURES], y_res, weight=weights, categorical_feature=CAT_COLS),
    num_boost_round=450)
p_res = m_res.predict(te_val.loc[te_open, FEATURES])
p2 = p_ses + p_res

# 50/50 log-space ensemble + Bias Calibration
# CV1 validated: +0.029 log-shift => 0.114 -> 0.110 RMSLE (-3.2%)
BIAS_SHIFT  = 0.029
final_log   = 0.5*p1 + 0.5*p2 + BIAS_SHIFT
final_open  = np.clip(np.expm1(final_log), 0, None)
print(f"Ensemble + bias calibration done. Total time: {time.time()-T0:.1f}s")

In [ ]:
# Build and validate submission
sub = te_val[["Id","IsOpen"]].copy()
sub["OrderVolume"] = 0.0
sub.loc[te_open, "OrderVolume"] = final_open
sub.loc[~te_open, "OrderVolume"] = 0.0

out = test[["Id"]].merge(sub[["Id","OrderVolume","IsOpen"]], on="Id", how="left")

assert out.shape[0] == 46830, f"Wrong rows: {out.shape[0]}"
assert not out["OrderVolume"].isna().any(), "NaN found!"
assert (out.loc[out["IsOpen"]==0, "OrderVolume"] == 0).all(), "Closed not zero!"
assert (out["OrderVolume"] >= 0).all(), "Negatives found!"

out[["Id","OrderVolume"]].to_csv("submission.csv", index=False)
print(f"submission.csv saved: {out.shape[0]} rows")
print(f"Closed zeros: {(out['IsOpen']==0).sum()}")
print(out["OrderVolume"].describe())